# Databricks/PySpark for Data Analysts/Engineers — Module 12: MLlib -- Basics

We're back to fully executable code. **MLlib** is Spark's machine learning library --
the same models we covered in the Statistics course (linear and logistic regression,
Modules 12-13), just now trained in a DISTRIBUTED way, on data too large to fit in a
single computer's memory.

## Table of contents
1. [MLlib: an overview of the DataFrame-based API](#sec1)
2. [Preparing features: VectorAssembler](#sec2)
3. [Categorical variables: StringIndexer and OneHotEncoder](#sec3)
4. [Pipeline: combining steps into one object](#sec4)
5. [Linear regression: predicting order value](#sec5)
6. [Logistic regression: predicting order cancellation](#sec6)
7. [Summary and exercises](#sec7)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. MLlib: an overview of the DataFrame-based API

Spark has had TWO ML libraries over its history: an old, RDD-based one (`pyspark.mllib`
-- DEPRECATED, don't use it in new code) and a newer, DataFrame-based one
(`pyspark.ml` -- the one we use in this module). Key concepts in `pyspark.ml`:

- **Transformer** -- an object that turns one DataFrame into another via a
  `.transform()` method (e.g. a trained model, a fitted `StringIndexer`).
- **Estimator** -- an object that LEARNS from data via a `.fit()` method and returns a
  Transformer (e.g. `LinearRegression` before training is an Estimator; after training
  it's a `LinearRegressionModel`, i.e. a Transformer).
- **Pipeline** -- a chain of Transformers and Estimators run in order as one coherent
  object (section 4).

> ⚡ **These are exactly the same models as in the Statistics course**
>
> The math underneath `LinearRegression`/`LogisticRegression` in MLlib is EXACTLY what we covered in Modules 12-13 of the Statistics course (`statsmodels`/`sklearn`) -- coefficients, R-squared, least-squares fitting, the logistic function. The only difference is that training happens in a DISTRIBUTED way across multiple machines -- interpreting the results stays identical.

<a id="sec2"></a>
## 2. Preparing features: VectorAssembler

MLlib models expect ONE vector-typed column (type `Vector`) holding all numeric
features at once -- unlike `sklearn`, where you pass a separate matrix `X`.
`VectorAssembler` combines selected columns into that one column (named `features` by
default).

In [ ]:
from pyspark.sql import functions as F

# Prepare data at the order level: item count, average price, total value
order_features = (
    orders.join(customers, "customer_id")
    .join(order_items, "order_id")
    .groupBy("order_id", "customer_id", "segment", "status")
    .agg(
        F.count("*").alias("item_count"),
        F.avg("unit_price").alias("avg_price"),
        F.sum(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))).alias("value"),
    )
)
order_features.show(5)


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["item_count", "avg_price"],
    outputCol="numeric_features",
)
with_features = assembler.transform(order_features)
with_features.select("order_id", "item_count", "avg_price", "numeric_features").show(5)


> ⚠️ **VectorAssembler doesn't accept NULL values**
>
> If any of the input columns has a `NULL`, `VectorAssembler` will raise an error at `.transform()`. Before assembling features into a vector, make sure gaps are handled (Module 3: `fillna`/`dropna`) -- exactly the same as when preparing data for `sklearn`/`statsmodels` in the Statistics course.

<a id="sec3"></a>
## 3. Categorical variables: StringIndexer and OneHotEncoder

ML models need numbers, not text -- a categorical column (e.g. `segment`) must first
be turned into numbers. Two steps: **`StringIndexer`** (assigns every unique text value
an integer, from most to least common) and **`OneHotEncoder`** (turns that integer into
a "one-hot" vector, so the model doesn't interpret consecutive numbers as a false
order/distance).

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

segment_indexer = StringIndexer(inputCol="segment", outputCol="segment_index")
with_index = segment_indexer.fit(order_features).transform(order_features)
with_index.select("segment", "segment_index").distinct().orderBy("segment_index").show()


In [ ]:
segment_encoder = OneHotEncoder(inputCol="segment_index", outputCol="segment_ohe")
with_ohe = segment_encoder.fit(with_index).transform(with_index)
with_ohe.select("segment", "segment_index", "segment_ohe").distinct().orderBy("segment_index").show()


> 🧱 **Why StringIndexer + OneHotEncoder, not one step**
>
> In `sklearn`, this is often done in one call (`OneHotEncoder` directly on text). MLlib deliberately splits this into TWO steps -- `StringIndexer` by itself is also useful on its own (e.g. `LogisticRegression` natively handles indexed class labels without one-hot encoding), so both steps are separate, freely composable building blocks in a `Pipeline` (section 4).

<a id="sec4"></a>
## 4. Pipeline: combining steps into one object

Instead of manually calling `.fit()`/`.transform()` for each step separately,
`Pipeline` combines ALL steps (indexers, encoders, assembler, model) into ONE object --
a single `.fit()` on the whole chain trains everything in order, and a single
`.transform()` runs new data through the entire chain.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder

indexer = StringIndexer(inputCol="segment", outputCol="segment_index")
encoder = OneHotEncoder(inputCol="segment_index", outputCol="segment_ohe")
assembler = VectorAssembler(
    inputCols=["item_count", "avg_price", "segment_ohe"],
    outputCol="features",
)

prep_pipeline = Pipeline(stages=[indexer, encoder, assembler])
prep_model = prep_pipeline.fit(order_features)
ready_data = prep_model.transform(order_features)
ready_data.select("order_id", "features", "value").show(5, truncate=False)


> ⚡ **Pipeline guarantees CONSISTENCY between training and new data**
>
> The biggest benefit of `Pipeline` isn't just convenience -- it guarantees that EXACTLY the same transformations (the same `StringIndexer` indices, the same `VectorAssembler` columns) get applied to NEW data in production as during training. Manually repeating these steps separately is a common source of bugs -- *train/serve skew*, where the model sees data in production prepared differently than during training.

<a id="sec5"></a>
## 5. Linear regression: predicting order value

Let's build a full `Pipeline` with a `LinearRegression` model at the end, predicting an
order's `value` from item count, average price, and customer segment -- exactly the
same model as in Module 12 of the Statistics course, now trained distributedly.

In [ ]:
from pyspark.ml.regression import LinearRegression

train, test = order_features.randomSplit([0.8, 0.2], seed=42)

reg_model = LinearRegression(featuresCol="features", labelCol="value")
reg_pipeline = Pipeline(stages=[indexer, encoder, assembler, reg_model])

trained_pipeline = reg_pipeline.fit(train)
predictions = trained_pipeline.transform(test)
predictions.select("order_id", "value", "prediction").show(10)


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="value", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.setMetricName("r2").evaluate(predictions)
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.3f}")


In [ ]:
# Model coefficients -- the same linear model from Module 12 of the Statistics course,
# just now living inside a Pipeline
final_model = trained_pipeline.stages[-1]
print(f"Coefficients: {final_model.coefficients}")
print(f"Intercept: {final_model.intercept:.2f}")


<a id="sec6"></a>
## 6. Logistic regression: predicting order cancellation

The second model from the Statistics course (Module 13) -- binary classification. We'll
predict whether an order gets CANCELLED, based on the same features. Our dataset has a
DELIBERATELY built-in relationship: customers in the `'New'` segment cancel much more
often (Module 3 -- here we finally put it to use in a model).

In [ ]:
from pyspark.ml.classification import LogisticRegression

labeled_orders = order_features.withColumn(
    "is_cancelled", (F.col("status") == "Cancelled").cast("double")
)

train_log, test_log = labeled_orders.randomSplit([0.8, 0.2], seed=42)

log_model = LogisticRegression(featuresCol="features", labelCol="is_cancelled")
log_pipeline = Pipeline(stages=[indexer, encoder, assembler, log_model])

trained_log = log_pipeline.fit(train_log)
log_predictions = trained_log.transform(test_log)
log_predictions.select("order_id", "is_cancelled", "probability", "prediction").show(10, truncate=False)


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

log_evaluator = BinaryClassificationEvaluator(
    labelCol="is_cancelled", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
auc = log_evaluator.evaluate(log_predictions)
print(f"AUC (area under the ROC curve): {auc:.3f}")


> ⚠️ **AUC from Module 9/13 of the Statistics course -- the same metric, read the same way**
>
> `areaUnderROC` is EXACTLY the same AUC metric we covered with logistic regression in the Statistics course -- 0.5 means the model is no better than random guessing, 1.0 means perfect class separation. The interpretation is identical whether the model was trained in `sklearn` or in MLlib.

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- `pyspark.ml` (DataFrame-based) vs. deprecated `pyspark.mllib` (RDD-based),
- the Transformer/Estimator/Pipeline concepts,
- `VectorAssembler` -- combining numeric columns into one `features` column,
- `StringIndexer` + `OneHotEncoder` -- encoding categorical variables,
- `Pipeline` -- combining all steps into one coherent, repeatable object,
- `LinearRegression` and `LogisticRegression` in MLlib -- the same models as in the Statistics course, trained distributedly.

### 📝 Exercise 1: VectorAssembler on products

Use `VectorAssembler` to assemble the `price` column from `products` into a `features` column (a single-element vector).

<details>
<summary>Show solution</summary>

```python
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["price"], outputCol="features")
assembler.transform(products).select("product_id", "price", "features").show(5)
```

</details>

### 📝 Exercise 2: StringIndexer on product category

Use `StringIndexer` on the `category` column from `products` and print the category-to-index assignment (sorted by index).

<details>
<summary>Show solution</summary>

```python
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="category", outputCol="category_index")
indexer.fit(products).transform(products).select(
    "category", "category_index"
).distinct().orderBy("category_index").show()
```

</details>

### 📝 Exercise 3: A simple regression Pipeline

Build a `Pipeline` with `VectorAssembler` (feature: `item_count`) and `LinearRegression` (`labelCol="value"`) on `order_features`, train it on the whole dataset (no train/test split), and print `coefficients`/`intercept`.

<details>
<summary>Show solution</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(inputCols=["item_count"], outputCol="features")
model = LinearRegression(featuresCol="features", labelCol="value")
pipeline = Pipeline(stages=[assembler, model])

trained = pipeline.fit(order_features)
final = trained.stages[-1]
print(f"Coefficients: {final.coefficients}")
print(f"Intercept: {final.intercept:.2f}")
```

</details>

### 📝 Exercise 4: RMSE evaluation without a train/test split

Train the `Pipeline` from exercise 3, run the WHOLE `order_features` dataset through it, and compute RMSE via `RegressionEvaluator`.

<details>
<summary>Show solution</summary>

```python
from pyspark.ml.evaluation import RegressionEvaluator

predictions = trained.transform(order_features)
evaluator = RegressionEvaluator(labelCol="value", predictionCol="prediction", metricName="rmse")
print(f"RMSE: {evaluator.evaluate(predictions):.2f}")
```

</details>

> 🔥 **Challenge: a full Pipeline with multiple features and an AUC comparison**
>
> Build TWO `LogisticRegression` models predicting order cancellation: (1) using only the `item_count` feature, (2) using the full feature set from section 6 (`item_count`, `avg_price`, `segment` via one-hot encoding). Compare the AUC of both models on the same test set -- does adding segment improve the model?

<details>
<summary>Show solution</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

data = order_features.withColumn(
    "is_cancelled", (F.col("status") == "Cancelled").cast("double")
)
train, test = data.randomSplit([0.8, 0.2], seed=42)

# Model 1: item_count only
simple_assembler = VectorAssembler(inputCols=["item_count"], outputCol="features")
simple_model = LogisticRegression(featuresCol="features", labelCol="is_cancelled")
simple_pipeline = Pipeline(stages=[simple_assembler, simple_model])
simple_predictions = simple_pipeline.fit(train).transform(test)

# Model 2: full feature set
indexer = StringIndexer(inputCol="segment", outputCol="segment_index")
encoder = OneHotEncoder(inputCol="segment_index", outputCol="segment_ohe")
full_assembler = VectorAssembler(
    inputCols=["item_count", "avg_price", "segment_ohe"], outputCol="features"
)
full_model = LogisticRegression(featuresCol="features", labelCol="is_cancelled")
full_pipeline = Pipeline(stages=[indexer, encoder, full_assembler, full_model])
full_predictions = full_pipeline.fit(train).transform(test)

evaluator = BinaryClassificationEvaluator(
    labelCol="is_cancelled", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
print(f"AUC (item_count only): {evaluator.evaluate(simple_predictions):.3f}")
print(f"AUC (full feature set): {evaluator.evaluate(full_predictions):.3f}")
```

Since our dataset has a deliberately built-in segment-cancellation relationship (Module 3), the model with segment should achieve a higher AUC -- exactly what we'd expect from a good predictive feature.

</details>

## What's next?

In **Module 13** we'll cover more rigorous model evaluation: train/test/validation
splits, `CrossValidator` and `ParamGridBuilder` for hyperparameter tuning -- a fully
automated search for the best model configuration.